# 03 – Replay months
Simulates a live source system: takes one month of the static Olist export from
`landing/source` and drops it into `landing/raw`, as if new data had just arrived.

**Modes**
- `range` – manual backfill of an explicit month range (`from_month` → `to_month`)
- `next` – used by the Job: replays exactly one month, the next one not yet in landing

**Commit marker:** the `orders` file of a month is written last. A month counts as
replayed only when its `orders` file exists.

## 1. Imports and parameters

In [0]:
import os
import re

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# mode=range -> manual backfill of an explicit month range (original behaviour)
# mode=next  -> used by the Job: a Job cannot type months into widgets every day,
#               so the notebook must decide by itself which month comes next
dbutils.widgets.text("catalog", "olist_dev", "Target catalog")
dbutils.widgets.dropdown("mode", "range", ["range", "next"], "Mode")
dbutils.widgets.text("from_month", "2016-09", "From month (yyyy-MM, mode=range)")
dbutils.widgets.text("to_month", "2017-12", "To month (yyyy-MM, mode=range)")

CATALOG = dbutils.widgets.get("catalog")
MODE = dbutils.widgets.get("mode")
FROM_MONTH = dbutils.widgets.get("from_month")
TO_MONTH = dbutils.widgets.get("to_month")

print(f"catalog={CATALOG}, mode={MODE}")

## 2. Paths and constants

In [0]:
SOURCE = f"/Volumes/{CATALOG}/landing/source/olist"
LANDING = f"/Volumes/{CATALOG}/landing/raw/olist"
CSV_OPTIONS = {"header": "true", "multiLine": "true", "escape": '"'}

# Tables without a date column; they are split by the order_id of monthly orders
RELATED_TABLES = ["order_items", "order_payments", "order_reviews"]

# orders_2017_01.csv -> captures "2017" and "01".
# The orders file is the commit marker of a month (written last, see step 5),
# so it is the only file we need to check to know which months are done.
ORDERS_FILE_PATTERN = re.compile(r"^orders_(\d{4})_(\d{2})\.csv$")
MONTH_PATTERN = r"^\d{4}-(0[1-9]|1[0-2])$"

## 3. Helper functions
- `read_source` – reads a full table extract from the source volume
- `write_single_csv` – writes one monthly slice as a single CSV file
- `get_replayed_months` – which months are already in landing (from file names)
- `resolve_months` – which months to replay in this run, based on the mode

In [0]:
def read_source(table: str) -> DataFrame:
    """Read the full extract of one table from the source volume (all columns as strings)."""
    return spark.read.options(**CSV_OPTIONS).csv(f"{SOURCE}/{table}/")


def write_single_csv(df: DataFrame, table: str, month: str) -> int:
    """Write a monthly slice as one CSV file into landing and return the row count.

    Uses pandas because Spark writes a folder of part files, not a single file.
    Example: /Volumes/olist_dev/landing/raw/olist/orders/orders_2017_01.csv
    """
    target_dir = f"{LANDING}/{table}"
    os.makedirs(target_dir, exist_ok=True)
    path = f"{target_dir}/{table}_{month.replace('-', '_')}.csv"

    pdf = df.toPandas()  # compute once; reuse for writing and counting
    pdf.to_csv(path, index=False)
    return len(pdf)


def get_replayed_months() -> list[str]:
    """Months already replayed into landing, derived from orders file names (yyyy-MM).

    Files are the single source of truth: no extra state table that could drift
    from reality (e.g. after someone deletes a file by hand).
    """
    orders_dir = f"{LANDING}/orders"
    if not os.path.exists(orders_dir):
        return []  # very first run: nothing replayed yet
    months = []
    for file_name in os.listdir(orders_dir):
        match = ORDERS_FILE_PATTERN.match(file_name)
        if match:  # ignore anything that does not follow our naming
            months.append(f"{match.group(1)}-{match.group(2)}")
    return sorted(months)


def resolve_months(source_months: list[str]) -> list[str]:
    """Decide which months to replay in this run, based on the mode."""
    if MODE == "range":
        # Fail fast on invalid input instead of silently writing nothing
        for name, value in {"from_month": FROM_MONTH, "to_month": TO_MONTH}.items():
            if not re.match(MONTH_PATTERN, value):
                raise ValueError(f"{name} must be in format yyyy-MM, got '{value}'")
        if FROM_MONTH > TO_MONTH:
            raise ValueError(f"from_month ({FROM_MONTH}) must not be after to_month ({TO_MONTH})")
        months = [m for m in source_months if FROM_MONTH <= m <= TO_MONTH]
        if not months:
            raise ValueError(f"No orders found between {FROM_MONTH} and {TO_MONTH}")
        return months

    # MODE == "next"
    replayed = get_replayed_months()
    last = replayed[-1] if replayed else None
    print(f"Last replayed month: {last or 'none (first run)'}")

    # Next month that EXISTS in the source, not simply last + 1:
    # the source has gaps (sparse 2016), and last + 1 could be an empty month forever.
    # "yyyy-MM" strings sort correctly as text, so a plain > comparison is enough.
    candidates = [m for m in source_months if last is None or m > last]
    return candidates[:1]  # exactly one month per run = one simulated "day" of new data

## 4. Read source and choose months
Reads the full extracts once and decides which months this run will replay.
If the source is fully replayed, the notebook ends here **without an error**
(the Job stays green; ingest simply finds no new files)


In [0]:
# Timestamps look like "2017-10-02 10:56:33" -> first 7 characters = "yyyy-MM"
orders = read_source("orders").withColumn("_month", F.substring("order_purchase_timestamp", 1, 7))
related = {table: read_source(table) for table in RELATED_TABLES}

# All months present in the source, oldest first
source_months = [
    row["_month"]
    for row in orders.select("_month").distinct().orderBy("_month").collect()
]

months = resolve_months(source_months)
print(f"Months to replay: {months}")

if not months:
    # End of the source is not an error: ingest finds no new files (Auto Loader checkpoint)
    # and dbt rebuilds the same data. A failing task would only send a false alert.
    dbutils.notebook.exit("no new month")

## 5. Write monthly files
Related tables first, `orders` **last** (commit marker). If the run fails halfway,
the `orders` file is missing, so the next run redoes the whole month instead of
skipping it with incomplete data.

In [0]:
summary = []
for month in months:
    # Helper column _month must not end up in the exported file
    orders_month = orders.where(F.col("_month") == month).drop("_month")
    order_ids = orders_month.select("order_id")

    row = {"month": month}

    # left_semi = keep rows of df that have a matching order_id, add no columns
    for table, df in related.items():
        month_df = df.join(order_ids, on="order_id", how="left_semi")
        row[table] = write_single_csv(month_df, table, month)

    row["orders"] = write_single_csv(orders_month, "orders", month)  # marker written last
    summary.append(row)
    print(f"OK  {month}")

In [0]:
display(spark.createDataFrame(pd.DataFrame(summary)))

# Return value shown in the Job run (which month(s) this run produced)
dbutils.notebook.exit(",".join(months))